# 🚀 Task 15: End-to-End Machine Learning Pipeline
**Project:** Production-Ready Titanic Survival Prediction  
**Author:** [Your Name]

## 🎯 Objective
To build a robust **Scikit-Learn Pipeline** that automates the entire machine learning workflow:
1.  **Data Imputation:** Handling missing values automatically.
2.  **Transformation:** Scaling numbers and encoding categories.
3.  **Modeling:** Training a Random Forest classifier.

**Why Pipelines?**
In manual workflows, you often scale your training data but forget to scale your test data, or you scale them separately (causing data leakage). A Pipeline locks these steps together, ensuring the exact same transformations are applied to every new data point.

## 📂 Dataset
* **Source:** Titanic Dataset.
* **Challenge:** Mixed data types.
    * *Numeric:* Age, Fare (Needs scaling, has missing values).
    * *Categorical:* Sex, Embarked (Needs encoding, has missing values).

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Load Dataset (Titanic)
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)

# Select relevant features
# We deliberately pick mixed types to show off the pipeline capabilities
features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

X = df[features]
y = df[target]

# 2. Split Data
# Note: We are splitting RAW data. We haven't touched missing values or encoding yet.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data Split Complete.")
print(f"Train Shape: {X_train.shape}")
print(f"Test Shape:  {X_test.shape}")
X_train.head()

Data Split Complete.
Train Shape: (712, 7)
Test Shape:  (179, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [2]:
# 3. Define Transformers

# A. Numerical Pipeline:
#    1. Fill missing values with Median
#    2. Scale features (StandardScaler)
numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# B. Categorical Pipeline:
#    1. Fill missing values with 'S' (most frequent)
#    2. One-Hot Encode (Convert 'male' -> [0, 1])
categorical_features = ['Pclass', 'Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Combine into ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing Pipeline Created.")

Preprocessing Pipeline Created.


In [3]:
# 5. Create Final Pipeline (Preprocessor + Model)
# We use Random Forest, but you could swap this for LogisticRegression() easily
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 6. Train the Pipeline
# Notice we pass RAW X_train. The pipeline handles imputation, scaling, and encoding internally!
print("Training Pipeline...")
model_pipeline.fit(X_train, y_train)
print("Training Complete!")

Training Pipeline...
Training Complete!


In [4]:
# 7. Evaluate
y_pred = model_pipeline.predict(X_test)

print("--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# 8. Save the Entire Pipeline
joblib.dump(model_pipeline, 'titanic_pipeline.pkl')
print("\nPipeline saved as 'titanic_pipeline.pkl'")
print("You can now load this file and predict on raw data directly!")

--- Model Performance ---
Accuracy: 0.8156

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.86      0.85       105
           1       0.79      0.76      0.77        74

    accuracy                           0.82       179
   macro avg       0.81      0.81      0.81       179
weighted avg       0.81      0.82      0.82       179


Pipeline saved as 'titanic_pipeline.pkl'
You can now load this file and predict on raw data directly!


## 🧠 Conclusion

### 1. The Power of Pipelines
We successfully wrapped a complex process into a single object `model_pipeline`.
* **Before:** We would need 4-5 manual steps to clean new data before predicting.
* **Now:** We just run `model_pipeline.predict(new_data)`.

### 2. Model Performance
* **Accuracy:** Achieved ~80-82% accuracy.
* **Robustness:** Because we used `SimpleImputer` inside the pipeline, our model won't crash even if the new data has missing values.

### 3. Deployment
The saved file `titanic_pipeline.pkl` contains everything: the imputer statistics, the scaling math, the one-hot encoding maps, and the random forest trees. This file is ready to be loaded into a Streamlit app or Flask API.